# Dhaka Air Quality Dataset — Data Collection (Project E1)

This notebook downloads a large, multi-year **hourly** dataset for Dhaka and
saves it as a single CSV. It combines:

* **Air pollution** (PM2.5, PM10, NO2, CO, O3, SO2) from the OpenAQ API
* **Weather** (temperature, wind, humidity, pressure, rainfall, etc.) from the Open-Meteo ERA5 archive
* **Traffic indicators** derived from the calendar (rush hour, weekend, hour of day)

**How to use:** paste your free OpenAQ API key in the next cell, then run every
cell in order (Runtime → Run all). The final cell downloads the dataset.

> Note: OpenAQ does not provide CO2 (carbon dioxide). It provides **CO**
> (carbon monoxide), a traffic-related pollutant, which is collected here.

## Step 0 — Settings (paste your API key here)

In [15]:
# Get a FREE key in 2 minutes: register at https://explore.openaq.org
# then copy the key from your account page and paste it between the quotes.
OPENAQ_API_KEY = "3e86e5d5f8817a6dc25173b16c7f70a4f04da94a09aa1fd31188bd1f6ae1a7bc"          # <-- PASTE YOUR KEY HERE

# Study area: central Dhaka, Bangladesh
DHAKA_LAT, DHAKA_LON = 23.7806, 90.4193
RADIUS_M = 25000             # search radius for monitoring stations (25 km)

# Time span. The notebook collects as much as is available in this window.
START_DATE = "2013-01-01"
END_DATE   = "2026-05-20"

# Pollutants to collect (every criteria pollutant OpenAQ reports).
PARAMETERS = ["pm25", "pm10", "no2", "co", "o3", "so2"]

assert OPENAQ_API_KEY, "3e86e5d5f8817a6dc25173b16c7f70a4f04da94a09aa1fd31188bd1f6ae1a7bc"
print("Settings OK. Collecting data for Dhaka,",
      START_DATE, "to", END_DATE)

Settings OK. Collecting data for Dhaka, 2013-01-01 to 2026-05-20


## Step 1 — Download air pollution data (OpenAQ)

This finds every monitoring station within 25 km of Dhaka, then downloads
hourly pollutant readings for each one. Requests are split one year at a time
and the OpenAQ rate limit (60 requests/minute) is handled automatically, so
this step may take several minutes.

In [16]:
import requests, time
import pandas as pd
from datetime import datetime, timezone
from dateutil.relativedelta import relativedelta

BASE = "https://api.openaq.org/v3"
HEADERS = {"X-API-Key": OPENAQ_API_KEY}

class OpenAQError(Exception):
    """Raised when an OpenAQ request fails permanently. Carries status + body
    so the caller can see exactly why it failed and decide whether to skip."""

def openaq_get(path, params):
    """Call an OpenAQ v3 endpoint with retry + rate-limit handling."""
    last_status, last_body = None, ""
    for attempt in range(6):
        try:
            r = requests.get(f"{BASE}/{path}", headers=HEADERS,
                              params=params, timeout=60)
        except requests.RequestException as e:
            last_status, last_body = "network-error", str(e)
            time.sleep(2 ** attempt); continue
        last_status, last_body = r.status_code, r.text[:300]
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:                       # rate limited
            wait = int(r.headers.get("X-RateLimit-Reset", 60)) + 1
            print(f"  rate limit reached; waiting {wait}s ...")
            time.sleep(wait); continue
        if r.status_code in (408, 500, 502, 503, 504): # transient
            time.sleep(2 ** attempt); continue
        # Any other 4xx (404 not found, 410 gone, 422 validation, etc.) is
        # permanent — no point retrying. Stop now and report the real reason.
        break
    raise OpenAQError(
        f"{path} failed (status={last_status}): {last_body}")

# --- 1a. Find every sensor near Dhaka for the chosen pollutants ---
sensors, page = [], 1
while True:
    data = openaq_get("locations", {
        "coordinates": f"{DHAKA_LAT},{DHAKA_LON}",
        "radius": RADIUS_M, "limit": 1000, "page": page})
    results = data.get("results", [])
    for loc in results:
        for s in loc.get("sensors", []):
            pname = (s.get("parameter") or {}).get("name")
            if pname in PARAMETERS:
                sensors.append({"sensor_id": s["id"], "parameter": pname,
                                "station": loc.get("name")})
    if len(results) < 1000:
        break
    page += 1
sensors = pd.DataFrame(sensors).drop_duplicates("sensor_id")
print(f"Found {len(sensors)} sensors across Dhaka:")
print(sensors.to_string(index=False))

# --- 1b. Download hourly values for each sensor, one year per request.
#         A single bad sensor must NOT abort the whole collection. ---
rows, failed = [], []
for n, s in enumerate(sensors.itertuples(), 1):
    print(f"[{n}/{len(sensors)}] sensor {s.sensor_id} ({s.parameter}) ...")
    start = datetime.fromisoformat(START_DATE).replace(tzinfo=timezone.utc)
    final = datetime.fromisoformat(END_DATE).replace(tzinfo=timezone.utc)
    sensor_ok = True
    while start < final and sensor_ok:
        chunk_end = min(start + relativedelta(years=1), final)
        page = 1
        while True:
            try:
                data = openaq_get(f"sensors/{s.sensor_id}/hours", {
                    # OpenAQ v3 wants the Z-suffix ISO form, not "+00:00".
                    "datetime_from": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
                    "datetime_to":   chunk_end.strftime("%Y-%m-%dT%H:%M:%SZ"),
                    "limit": 1000, "page": page})
            except OpenAQError as e:
                print(f"  -> skipping sensor {s.sensor_id}: {e}")
                failed.append((s.sensor_id, s.parameter, str(e)))
                sensor_ok = False
                break
            results = data.get("results", [])
            for r in results:
                dt = (r.get("period", {}).get("datetimeFrom") or {}).get("utc")
                rows.append({"datetime_utc": dt, "parameter": s.parameter,
                             "value": r.get("value")})
            if len(results) < 1000:
                break
            page += 1
        start = chunk_end
        time.sleep(0.5)

if failed:
    print(f"\n{len(failed)} sensor(s) skipped:")
    for sid, p, msg in failed:
        print(f"  sensor {sid} ({p}): {msg}")

# --- 1c. Reshape into a wide hourly table (one column per pollutant) ---
aq = pd.DataFrame(rows)
if aq.empty:
    raise RuntimeError(
        "No air-quality rows were collected. Check the API key, network, "
        "and the 'skipped' messages above.")
aq["datetime_utc"] = pd.to_datetime(aq["datetime_utc"], utc=True)
aq = (aq.dropna(subset=["datetime_utc"])
        .groupby(["datetime_utc", "parameter"])["value"].mean()
        .unstack("parameter").sort_index())
aq.columns = [f"aq_{c}" for c in aq.columns]
aq.to_csv("dhaka_air_quality_raw.csv")
print(f"\nAir pollution data: {aq.shape[0]} hourly rows, "
      f"columns: {list(aq.columns)}")

Found 24 sensors across Dhaka:
 sensor_id parameter                                                          station
        41      pm25                                       SPARTAN - Dhaka University
      4680      pm25                                        US Diplomatic Post: Dhaka
   7464180        o3                                                            Dhaka
     24434      pm25                                                            Dhaka
   6566258      pm25                                       SPARTAN - Dhaka University
  11106524      pm10 Dept. of Public Health and Informatics, Jahangirnagar University
  11106529      pm25 Dept. of Public Health and Informatics, Jahangirnagar University
  13623705      pm25                                      smart air bangladesh office
  14839069      pm25              RAJUK Uttara Apartment Project, Sector - 18, Uttara
  15296733      pm25                                         Abdullahbag, North Badda
  15297527      pm25   

## Step 2 — Download weather data (Open-Meteo ERA5)

This downloads the matching hourly weather record for Dhaka in a single
request. No API key is needed. ERA5 is a gap-free reanalysis dataset, so the
weather series has no missing hours.

In [17]:
import requests
import pandas as pd

WEATHER_VARS = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m",
    "wind_speed_10m", "wind_direction_10m", "surface_pressure",
    "precipitation", "cloud_cover", "boundary_layer_height",
    "shortwave_radiation",
]

resp = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
    "latitude": DHAKA_LAT, "longitude": DHAKA_LON,
    "start_date": START_DATE, "end_date": END_DATE,
    "hourly": ",".join(WEATHER_VARS), "timezone": "UTC"}, timeout=180)
resp.raise_for_status()
hourly = resp.json()["hourly"]

wx = pd.DataFrame(hourly)
wx["datetime_utc"] = pd.to_datetime(wx.pop("time"), utc=True)
wx = wx.set_index("datetime_utc").sort_index()
wx.columns = [f"met_{c}" for c in wx.columns]
wx.to_csv("dhaka_weather_raw.csv")
print(f"Weather data: {wx.shape[0]} hourly rows, {wx.shape[1]} variables")

Weather data: 117312 hourly rows, 10 variables


## Step 3 — Merge everything and add traffic features

This puts pollution and weather on one shared hourly timeline, adds
calendar-based traffic indicators (Dhaka's weekend is Friday–Saturday), and
saves the final dataset.

In [23]:
import numpy as np
import pandas as pd

# One continuous hourly timeline in UTC.
grid = pd.date_range(START_DATE, f"{END_DATE} 23:00", freq="h", tz="UTC")
df = wx.reindex(grid).join(aq.reindex(grid))
df.index.name = "datetime_utc"

# Calendar / traffic-proxy features (based on Dhaka local time).
local = df.index.tz_convert("Asia/Dhaka")
df["hour"] = local.hour
df["day_of_week"] = local.dayofweek                 # Monday=0 ... Sunday=6
df["month"] = local.month
df["is_weekend"] = local.dayofweek.isin([4, 5]).astype(int)   # Fri, Sat
df["is_rush_hour"] = local.hour.isin([8, 9, 10, 17, 18, 19, 20]).astype(int)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Convenience target for later modeling: PM2.5 one day (24 h) ahead.
if "aq_pm25" in df.columns:
    df["pm25_next_day"] = df["aq_pm25"].shift(-24)

df.to_csv("dhaka_air_quality_dataset.csv")
print(f"FINAL DATASET: {df.shape[0]} rows  x  {df.shape[1]} columns")
print("Saved as: dhaka_air_quality_dataset.csv\n")

# Quick coverage report so we can see how much real data you have.
print("Data completeness per column (fraction of hours with a value):")
print(df.notna().mean().round(3).to_string())
df.head()

FINAL DATASET: 117312 rows  x  21 columns
Saved as: dhaka_air_quality_dataset.csv

Data completeness per column (fraction of hours with a value):
met_temperature_2m           1.000
met_relative_humidity_2m     1.000
met_dew_point_2m             1.000
met_wind_speed_10m           1.000
met_wind_direction_10m       1.000
met_surface_pressure         1.000
met_precipitation            1.000
met_cloud_cover              1.000
met_boundary_layer_height    0.963
met_shortwave_radiation      1.000
aq_o3                        0.063
aq_pm10                      0.013
aq_pm25                      0.298
hour                         1.000
day_of_week                  1.000
month                        1.000
is_weekend                   1.000
is_rush_hour                 1.000
hour_sin                     1.000
hour_cos                     1.000
pm25_next_day                0.298


,met_temperature_2m,met_relative_humidity_2m,met_dew_point_2m,met_wind_speed_10m,met_wind_direction_10m,met_surface_pressure,met_precipitation,met_cloud_cover,met_boundary_layer_height,met_shortwave_radiation,...,aq_pm10,aq_pm25,hour,day_of_week,month,is_weekend,is_rush_hour,hour_sin,hour_cos,pm25_next_day
datetime_utc,,,,,,,,,,,,,,,,,,,,,
2013-01-01 00:00:00+00:00,11.6,96,11.0,6.9,354,1012.7,0.0,4,40.0,0.0,...,NaN,NaN,6,1,1,0,0,1.000000,6.123234e-17,NaN
2013-01-01 01:00:00+00:00,11.4,96,10.8,5.4,360,1013.5,0.0,4,40.0,3.0,...,NaN,NaN,7,1,1,0,0,0.965926,-2.588190e-01,NaN
2013-01-01 02:00:00+00:00,13.5,93,12.5,5.1,4,1014.1,0.0,1,60.0,107.0,...,NaN,NaN,8,1,1,0,1,0.866025,-5.000000e-01,NaN
2013-01-01 03:00:00+00:00,17.4,77,13.4,4.0,360,1015.0,0.0,1,110.0,292.0,...,NaN,NaN,9,1,1,0,1,0.707107,-7.071068e-01,NaN
2013-01-01 04:00:00+00:00,20.3,66,13.6,4.4,351,1015.3,0.0,1,240.0,482.0,...,NaN,NaN,10,1,1,0,1,0.500000,-8.660254e-01,NaN


## Step 4 — Save the dataset

The cell below downloads the dataset to my computer. Also, I am creating a copy and adding datatime as column and organizing it in a descending manner


In [31]:
import pandas as pd

try:
    df_out = df.copy()
except NameError:
    df_out = pd.read_csv("dhaka_air_quality_dataset.csv")

# Make sure datetime_utc is a real column (not just the index) and that
# pandas treats it as a UTC timestamp so the timezone math is correct.
if df_out.index.name == "datetime_utc":
    df_out = df_out.reset_index()
df_out["datetime_utc"] = pd.to_datetime(df_out["datetime_utc"], utc=True)

# Local-time versions, easy to read in Excel.
df_out["datetime_local"] = df_out["datetime_utc"].dt.tz_convert("Asia/Dhaka")
df_out["date"] = df_out["datetime_local"].dt.strftime("%Y-%m-%d")
df_out["time"] = df_out["datetime_local"].dt.strftime("%H:%M")

# Put the time columns first, then sort newest -> oldest.
front = ["date", "time", "datetime_local", "datetime_utc"]
df_out = df_out[front + [c for c in df_out.columns if c not in front]]
df_out = df_out.sort_values("datetime_utc", ascending=False).reset_index(drop=True)

df_out.to_csv("dhaka_air_quality_dataset.csv", index=False)
print(f"Organized: {df_out.shape[0]} rows x {df_out.shape[1]} columns")
print(f"Top row    : {df_out['date'].iloc[0]} {df_out['time'].iloc[0]}")
print(f"Bottom row : {df_out['date'].iloc[-1]} {df_out['time'].iloc[-1]}")
df_out.head()

Organized: 117312 rows x 25 columns
Top row    : 2026-05-21 05:00
Bottom row : 2013-01-01 06:00


,date,time,datetime_local,datetime_utc,met_temperature_2m,met_relative_humidity_2m,met_dew_point_2m,met_wind_speed_10m,met_wind_direction_10m,met_surface_pressure,...,aq_pm10,aq_pm25,hour,day_of_week,month,is_weekend,is_rush_hour,hour_sin,hour_cos,pm25_next_day
0,2026-05-21,05:00,2026-05-21 05:00:00+06:00,2026-05-20 23:00:00+00:00,27.6,92,26.1,8.1,152,1001.8,...,NaN,NaN,5,3,5,0,0,0.965926,0.258819,NaN
1,2026-05-21,04:00,2026-05-21 04:00:00+06:00,2026-05-20 22:00:00+00:00,27.7,91,26.1,8.9,155,1001.0,...,NaN,NaN,4,3,5,0,0,0.866025,0.500000,NaN
2,2026-05-21,03:00,2026-05-21 03:00:00+06:00,2026-05-20 21:00:00+00:00,27.9,90,26.1,9.4,145,1000.4,...,NaN,NaN,3,3,5,0,0,0.707107,0.707107,NaN
3,2026-05-21,02:00,2026-05-21 02:00:00+06:00,2026-05-20 20:00:00+00:00,28.1,91,26.4,8.5,152,1000.4,...,NaN,NaN,2,3,5,0,0,0.500000,0.866025,NaN
4,2026-05-21,01:00,2026-05-21 01:00:00+06:00,2026-05-20 19:00:00+00:00,28.3,91,26.6,8.2,158,1000.8,...,NaN,NaN,1,3,5,0,0,0.258819,0.965926,NaN
